# AIA on SDO — Implementation / AIA on SDO 구현

**논문 / Paper**: Lemen, J.R., et al. (2012). "The Atmospheric Imaging Assembly (AIA) on the Solar Dynamics Observatory (SDO)." *Solar Physics*, 275, 17–40. [DOI: 10.1007/s11207-011-9776-8]

AIA는 SDO 위성에 탑재된 4개의 망원경으로 구성된 EUV/UV 태양 영상 장비이다. 10개 채널, 4096×4096 CCD, 0.6"/pixel 해상도, 41' 시야각, 12초 관측 주기를 갖는다.

AIA is a suite of 4 telescopes on SDO providing EUV/UV solar imaging. It features 10 channels, 4096×4096 CCD, 0.6"/pixel resolution, 41' FOV, and 12-second cadence.

이 노트북은 AIA의 핵심 설계 개념과 성능을 재현하고 분석한다.

This notebook reproduces and analyzes the key design concepts and performance of AIA.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Circle
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
from scipy.integrate import trapezoid

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

---
## Part 1: AIA Temperature Response Functions / AIA 온도 응답 함수

AIA의 7개 EUV 채널 온도 응답 함수 $K_i(T)$를 Gaussian 근사로 모델링한다. 이는 논문의 Fig. 13을 재현하는 핵심 그림이다.

We model the 7 EUV channel temperature response functions $K_i(T)$ using Gaussian approximations. This reproduces the key Figure 13 from the paper.

각 채널의 응답 함수는 특성 이온의 생성 온도에서 최대값을 가진다:

Each channel's response function peaks at the formation temperature of its characteristic ion:

$$K_i(T) = A_i \exp\left(-\frac{(\log T - \log T_i)^2}{2\sigma_i^2}\right)$$

In [ ]:
# Define AIA EUV channel parameters from Table 1 of Lemen et al. (2012).
# Each channel has: wavelength, ion, characteristic log T, peak response,
# Gaussian width, and plot color.

aia_channels = {
    "94":  {"ion": "Fe XVIII",  "logT": 6.8, "peak": 5e-27, "sigma": 0.25,
            "color": "green"},
    "131": {"ion": "Fe VIII/XXI", "logT": [5.6, 7.0], "peak": [1e-26, 2e-26],
            "sigma": [0.20, 0.20], "color": "teal"},
    "171": {"ion": "Fe IX",     "logT": 5.8, "peak": 2e-25, "sigma": 0.20,
            "color": "gold"},
    "193": {"ion": "Fe XII/XXIV", "logT": [6.2, 7.3], "peak": [6e-26, 3e-26],
            "sigma": [0.25, 0.20], "color": "brown"},
    "211": {"ion": "Fe XIV",    "logT": 6.3, "peak": 4e-26, "sigma": 0.22,
            "color": "purple"},
    "304": {"ion": "He II",     "logT": 4.7, "peak": 5e-27, "sigma": 0.25,
            "color": "red"},
    "335": {"ion": "Fe XVI",    "logT": 6.4, "peak": 3e-27, "sigma": 0.25,
            "color": "blue"},
}


def response_function(log_t, channel_params):
    """Compute the temperature response function for an AIA channel.

    Uses single or dual Gaussian approximation.

    Args:
        log_t: Array of log10(T) values.
        channel_params: Dictionary with logT, peak, sigma (scalar or list).

    Returns:
        Array of response values in DN/s/pixel per unit EM.
    """
    log_t_peaks = channel_params["logT"]
    peaks = channel_params["peak"]
    sigmas = channel_params["sigma"]

    # Ensure list format for dual-peak channels.
    if not isinstance(log_t_peaks, list):
        log_t_peaks = [log_t_peaks]
        peaks = [peaks]
        sigmas = [sigmas]

    result = np.zeros_like(log_t, dtype=float)
    for lt, pk, sg in zip(log_t_peaks, peaks, sigmas):
        result += pk * np.exp(-((log_t - lt) ** 2) / (2 * sg ** 2))
    return result

In [ ]:
# Plot (a): Full temperature response functions — reproducing Fig. 13 style.
log_t = np.linspace(4.0, 8.0, 1000)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Panel (a): Absolute response ---
ax = axes[0]
for wl, params in aia_channels.items():
    resp = response_function(log_t, params)
    ax.semilogy(log_t, resp, color=params["color"], lw=2,
                label=f"{wl} A ({params['ion']})")

ax.set_xlabel("log T [K]")
ax.set_ylabel(r"$K(T)$  [DN s$^{-1}$ pixel$^{-1}$ cm$^5$]")
ax.set_title("AIA EUV Temperature Response Functions (cf. Fig. 13)")
ax.set_xlim(4.0, 8.0)
ax.set_ylim(1e-28, 1e-23)
ax.legend(fontsize=9, loc="upper right")

# --- Panel (b): Normalized response ---
ax = axes[1]
for wl, params in aia_channels.items():
    resp = response_function(log_t, params)
    resp_norm = resp / np.max(resp)
    ax.plot(log_t, resp_norm, color=params["color"], lw=2,
            label=f"{wl} A")

ax.set_xlabel("log T [K]")
ax.set_ylabel("Normalized Response")
ax.set_title("Normalized AIA Response — Temperature Sensitivity")
ax.set_xlim(4.0, 8.0)
ax.set_ylim(0, 1.1)
ax.legend(fontsize=9, loc="upper right", ncol=2)

plt.tight_layout()
plt.show()

print("171 A (Fe IX) has the strongest response — dominant in quiet Sun imaging.")
print("131 A and 193 A show dual peaks: sensitivity to both cool and very hot plasma.")
print("304 A (He II) probes the chromosphere/transition region at ~50,000 K.")

---
## Part 2: DEM Reconstruction Concept / DEM 역문제 개념

AIA의 다중 채널 관측은 코로나 플라즈마의 차등 방출 척도(DEM)를 추정하는 데 사용된다. 순방향 문제는 다음과 같다:

AIA's multi-channel observations are used to estimate the coronal differential emission measure (DEM). The forward problem is:

$$g_i = \int K_i(T) \cdot \text{DEM}(T) \, dT$$

여기서 $g_i$는 채널 $i$의 관측 강도(DN/s), $K_i(T)$는 온도 응답 함수, DEM(T)는 시선 방향의 온도별 방출 척도이다.

where $g_i$ is the observed intensity in channel $i$, $K_i(T)$ is the temperature response, and DEM(T) is the line-of-sight differential emission measure.

In [ ]:
def make_dem(log_t, feature="quiet_sun"):
    """Create a synthetic DEM profile for a given solar feature.

    Args:
        log_t: Array of log10(T) values.
        feature: One of 'quiet_sun', 'active_region', 'flare'.

    Returns:
        DEM array in cm^-5 K^-1.
    """
    dem = np.zeros_like(log_t, dtype=float)
    if feature == "quiet_sun":
        # Broad peak near log T ~ 6.15, EM ~ 10^21.
        dem += 1e21 * np.exp(-((log_t - 6.15) ** 2) / (2 * 0.15 ** 2))
    elif feature == "active_region":
        # Broader, hotter, stronger.
        dem += 1e22 * np.exp(-((log_t - 6.3) ** 2) / (2 * 0.25 ** 2))
    elif feature == "flare":
        # Two components: pre-flare AR + very hot flare.
        dem += 1e22 * np.exp(-((log_t - 6.3) ** 2) / (2 * 0.20 ** 2))
        dem += 1e24 * np.exp(-((log_t - 7.0) ** 2) / (2 * 0.15 ** 2))
    return dem


def forward_model(log_t, dem, channels):
    """Compute predicted AIA DN/s for each channel given a DEM.

    Uses trapezoidal integration: g_i = integral K_i(T) * DEM(T) dT.

    Args:
        log_t: Array of log10(T) values.
        dem: DEM array in cm^-5 K^-1.
        channels: Dictionary of channel parameters.

    Returns:
        Dictionary mapping wavelength string to predicted DN/s.
    """
    t_vals = 10.0 ** log_t
    predictions = {}
    for wl, params in channels.items():
        resp = response_function(log_t, params)
        integrand = resp * dem
        predictions[wl] = np.trapezoid(integrand, t_vals)
    return predictions

In [ ]:
# (a) Plot synthetic DEMs for three solar features.
log_t_fine = np.linspace(4.0, 8.0, 2000)
features = ["quiet_sun", "active_region", "flare"]
feature_labels = ["Quiet Sun", "Active Region", "Flare"]
feature_colors = ["steelblue", "darkorange", "crimson"]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
for feat, lbl, clr in zip(features, feature_labels, feature_colors):
    dem = make_dem(log_t_fine, feat)
    mask = dem > 0
    ax.semilogy(log_t_fine[mask], dem[mask], lw=2, color=clr, label=lbl)

ax.set_xlabel("log T [K]")
ax.set_ylabel(r"DEM [cm$^{-5}$ K$^{-1}$]")
ax.set_title("Synthetic DEM Profiles")
ax.set_xlim(4.5, 8.0)
ax.set_ylim(1e18, 1e26)
ax.legend()

# (b)-(c) Predicted channel intensities (fingerprints).
ax = axes[1]
wavelengths = list(aia_channels.keys())
x_pos = np.arange(len(wavelengths))
bar_width = 0.25

for i, (feat, lbl, clr) in enumerate(zip(features, feature_labels,
                                          feature_colors)):
    dem = make_dem(log_t_fine, feat)
    preds = forward_model(log_t_fine, dem, aia_channels)
    vals = [max(preds[wl], 1e-50) for wl in wavelengths]
    ax.bar(x_pos + i * bar_width, vals, bar_width, color=clr, label=lbl,
           alpha=0.8)

ax.set_yscale("log")
ax.set_xlabel("AIA Channel [A]")
ax.set_ylabel("Predicted DN/s (arbitrary units)")
ax.set_title("Channel Fingerprints for Different Solar Features")
ax.set_xticks(x_pos + bar_width)
ax.set_xticklabels([f"{wl}" for wl in wavelengths])
ax.legend()

plt.tight_layout()
plt.show()

# Print numerical predictions.
print("Predicted relative DN/s per channel:")
print(f"{'Channel':<10} {'Quiet Sun':>12} {'Active Region':>14} {'Flare':>12}")
print("-" * 50)
for wl in wavelengths:
    qs = forward_model(log_t_fine, make_dem(log_t_fine, "quiet_sun"),
                       aia_channels)[wl]
    ar = forward_model(log_t_fine, make_dem(log_t_fine, "active_region"),
                       aia_channels)[wl]
    fl = forward_model(log_t_fine, make_dem(log_t_fine, "flare"),
                       aia_channels)[wl]
    print(f"{wl + ' A':<10} {qs:>12.3e} {ar:>14.3e} {fl:>12.3e}")

**해석 / Interpretation:**

- 조용한 태양(Quiet Sun): 171 A 채널이 지배적 — 약 100만 K 코로나에 대한 높은 감도를 반영한다.
- 활동 영역(Active Region): 193/211 A가 강해짐 — 더 뜨거운 플라즈마(2 MK)를 감지한다.
- 플레어(Flare): 94/131 A 채널이 급격히 증가 — 10 MK 이상의 고온 플라즈마에 반응한다.

- Quiet Sun: 171 A dominates — reflects high sensitivity to ~1 MK corona.
- Active Region: 193/211 A strengthen — detecting hotter plasma (~2 MK).
- Flare: 94/131 A channels surge — responding to >10 MK hot plasma.

이러한 서로 다른 "지문(fingerprint)"이 바로 7개 EUV 채널이 온도 진단을 가능하게 하는 이유이다.

These different fingerprints are exactly why 7 EUV channels enable temperature diagnostics.

---
## Part 3: AIA vs Predecessors — Complete Comparison / AIA vs 전임자 비교

AIA를 EIT, TRACE, EUVI와 정량적으로 비교한다. 해상도, 시야각, 채널 수, 데이터 속도 등 모든 면에서 AIA가 혁신적인 진보를 이룬 것을 보여준다.

We quantitatively compare AIA with EIT, TRACE, and EUVI. AIA represents a revolutionary advance in resolution, FOV, channel count, and data rate.

In [ ]:
# Instrument parameters for comparison.
instruments = {
    "EIT":  {"year": 1995, "pixel": 2.6, "fov": 45, "channels": 4,
             "ccd": 1024, "cadence": 720, "color": "C0"},
    "TRACE": {"year": 1998, "pixel": 0.5, "fov": 8.5, "channels": 8,
              "ccd": 1024, "cadence": 30, "color": "C1"},
    "EUVI": {"year": 2006, "pixel": 1.6, "fov": 54, "channels": 4,
             "ccd": 2048, "cadence": 150, "color": "C2"},
    "AIA":  {"year": 2010, "pixel": 0.6, "fov": 41, "channels": 10,
             "ccd": 4096, "cadence": 12, "color": "C3"},
}

KM_PER_ARCSEC = 725  # Physical scale on the Sun.

# (a) Resolution evolution.
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
names = list(instruments.keys())
years = [instruments[n]["year"] for n in names]
pixels = [instruments[n]["pixel"] for n in names]
km_res = [p * KM_PER_ARCSEC for p in pixels]

colors = [instruments[n]["color"] for n in names]
ax.bar(names, pixels, color=colors, alpha=0.7, edgecolor="black")
for i, (n, p, km) in enumerate(zip(names, pixels, km_res)):
    ax.text(i, p + 0.05, f'{p}"\n({km:.0f} km)', ha="center", fontsize=9)
ax.set_ylabel('Pixel Scale [arcsec]')
ax.set_title("Spatial Resolution Evolution")
ax.set_ylim(0, 3.5)

# (b) FOV vs Resolution scatter with bubble = channels.
ax = axes[1]
for name, params in instruments.items():
    ax.scatter(params["pixel"], params["fov"],
               s=params["channels"] * 80,
               color=params["color"], edgecolors="black",
               alpha=0.7, zorder=5)
    ax.annotate(f"{name}\n({params['channels']} ch)",
                (params["pixel"], params["fov"]),
                textcoords="offset points", xytext=(15, 5), fontsize=9)

ax.set_xlabel('Pixel Scale [arcsec]')
ax.set_ylabel("FOV [arcmin]")
ax.set_title("FOV vs Resolution (bubble = channels)")
ax.set_xlim(0, 3.5)
ax.set_ylim(0, 65)
# Draw the ideal corner.
ax.axhspan(30, 65, xmin=0, xmax=0.3, alpha=0.1, color="green")
ax.text(0.3, 58, "Ideal: full-disk + high-res", fontsize=8, color="green")

# (c) Information rate comparison.
ax = axes[2]
# Rate = pixels^2 * channels * (1/cadence).
eit_rate = (1024**2) * 4 * (1 / 720)
rates = {}
for name, params in instruments.items():
    rate = (params["ccd"] ** 2) * params["channels"] * (1 / params["cadence"])
    rates[name] = rate / eit_rate

bars = ax.bar(list(rates.keys()), list(rates.values()),
              color=[instruments[n]["color"] for n in rates],
              alpha=0.7, edgecolor="black")
ax.set_yscale("log")
ax.set_ylabel("Information Rate (relative to EIT)")
ax.set_title("Data Throughput Evolution")
for bar, val in zip(bars, rates.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, val * 1.5,
            f"{val:.0f}x", ha="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.show()

print(f"AIA information rate: {rates['AIA']:.0f}x EIT baseline.")
print(f"TRACE was {rates['TRACE']:.0f}x EIT — but only partial FOV.")
print("AIA uniquely combines full-disk FOV with high resolution AND fast cadence.")

---
## Part 4: 4-Telescope Architecture / 4-망원경 구조

AIA는 4개의 독립적인 Cassegrain 망원경으로 구성된다. 각 망원경은 2개의 채널을 담당하며, 12초 주기 내에 교대로 관측한다. 이 설계가 어떻게 12초 cadence를 달성하는지 시각화한다.

AIA consists of 4 independent Cassegrain telescopes. Each telescope handles 2 channels, alternating within the 12-second cycle. We visualize how this design achieves the 12s cadence.

In [ ]:
# (a) Telescope-channel assignment schematic.
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("AIA 4-Telescope Channel Assignment", fontsize=13)

telescope_info = [
    ("T1", ["131 A (Fe VIII/XXI)", "335 A (Fe XVI)"], "#4CAF50"),
    ("T2", ["193 A (Fe XII/XXIV)", "211 A (Fe XIV)"], "#2196F3"),
    ("T3", ["171 A (Fe IX)", "1600 A (C IV+cont)"], "#FF9800"),
    ("T4", ["304 A (He II)", "94 A (Fe XVIII)"], "#F44336"),
]

for i, (name, channels, color) in enumerate(telescope_info):
    y_center = 8.0 - i * 2.2
    # Telescope box.
    box = FancyBboxPatch((0.5, y_center - 0.5), 2.5, 1.0,
                         boxstyle="round,pad=0.1", facecolor=color,
                         alpha=0.3, edgecolor=color, lw=2)
    ax.add_patch(box)
    ax.text(1.75, y_center, name, ha="center", va="center",
            fontsize=14, fontweight="bold")
    # Channel boxes.
    for j, ch in enumerate(channels):
        cx = 5.5 + j * 2.2
        chbox = FancyBboxPatch((cx - 1.0, y_center - 0.4), 2.0, 0.8,
                               boxstyle="round,pad=0.05", facecolor=color,
                               alpha=0.15, edgecolor=color, lw=1)
        ax.add_patch(chbox)
        ax.text(cx, y_center, ch, ha="center", va="center", fontsize=8)
    # Arrow from telescope to channels.
    ax.annotate("", xy=(4.3, y_center), xytext=(3.1, y_center),
                arrowprops=dict(arrowstyle="->", color=color, lw=1.5))

ax.text(5.0, 9.5, "Each telescope: 2 channels per 12s cycle",
        ha="center", fontsize=10, style="italic")

# (b) Timing diagram showing the 12-second observing cycle.
ax = axes[1]
ax.set_xlim(0, 13)
ax.set_ylim(-0.5, 5)
ax.set_xlabel("Time [seconds]")
ax.set_title("12-Second Observing Cycle", fontsize=13)
ax.set_yticks([0.5, 1.5, 2.5, 3.5])
ax.set_yticklabels(["T4", "T3", "T2", "T1"])

# Exposure + readout blocks for each telescope.
# Each telescope takes image A, reads out, takes image B, reads out.
cycle = [
    # (telescope_y, start, duration, label, color)
    # T1.
    (3.5, 0.0, 2.0, "131 A exp", "#4CAF50"),
    (3.5, 2.0, 1.5, "readout", "gray"),
    (3.5, 6.0, 2.0, "335 A exp", "#4CAF50"),
    (3.5, 8.0, 1.5, "readout", "gray"),
    # T2.
    (2.5, 0.5, 2.0, "193 A exp", "#2196F3"),
    (2.5, 2.5, 1.5, "readout", "gray"),
    (2.5, 6.5, 2.0, "211 A exp", "#2196F3"),
    (2.5, 8.5, 1.5, "readout", "gray"),
    # T3.
    (1.5, 1.0, 2.0, "171 A exp", "#FF9800"),
    (1.5, 3.0, 1.5, "readout", "gray"),
    (1.5, 7.0, 2.0, "1600 A exp", "#FF9800"),
    (1.5, 9.0, 1.5, "readout", "gray"),
    # T4.
    (0.5, 1.5, 2.0, "304 A exp", "#F44336"),
    (0.5, 3.5, 1.5, "readout", "gray"),
    (0.5, 7.5, 2.0, "94 A exp", "#F44336"),
    (0.5, 9.5, 1.5, "readout", "gray"),
]

for y, start, dur, label, color in cycle:
    alpha = 0.3 if color == "gray" else 0.6
    rect = plt.Rectangle((start, y - 0.35), dur, 0.7,
                          facecolor=color, alpha=alpha, edgecolor="black",
                          lw=0.8)
    ax.add_patch(rect)
    fontsize = 6 if len(label) > 8 else 7
    ax.text(start + dur / 2, y, label, ha="center", va="center",
            fontsize=fontsize)

# Mark the 12s cycle boundary.
ax.axvline(12, color="red", ls="--", lw=1.5, alpha=0.7)
ax.text(12.2, 4.5, "12s", color="red", fontsize=10)

plt.tight_layout()
plt.show()

print("CCD readout: 4 quadrants read simultaneously at 2 Mpix/s each")
print(f"  -> 4096x4096 = {4096**2/1e6:.1f} Mpix total")
print(f"  -> Each quadrant: {4096**2/4/1e6:.1f} Mpix at 2 Mpix/s = ~2.1s readout")

---
## Part 5: Data Pipeline Visualization / 데이터 파이프라인

AIA는 초당 약 223 Mbit의 원시 데이터를 생성하며, Rice 압축 후 ~67 Mbit/s로 감소된다. 데이터는 White Sands 지상국을 경유하여 Stanford의 JSOC에 보관된다.

AIA generates ~223 Mbit/s of raw data, reduced to ~67 Mbit/s after Rice compression. Data flows through White Sands ground station to Stanford's JSOC archive.

In [ ]:
# (a) Processing levels diagram.
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 5)
ax.axis("off")
ax.set_title("AIA Data Processing Pipeline", fontsize=14)

levels = [
    (1.5, 3.5, "Level 0\nRaw 16-bit FITS\n(telemetry decom)", "#ffcccc"),
    (5.0, 3.5, "Level 1\nDark, Flat, Despike\nBad pixel removal", "#ccffcc"),
    (8.5, 3.5, "Level 1.5\nRotation, Plate scale\nCo-alignment", "#ccccff"),
]

for x, y, text, color in levels:
    box = FancyBboxPatch((x - 1.3, y - 0.9), 2.6, 1.8,
                         boxstyle="round,pad=0.15", facecolor=color,
                         edgecolor="black", lw=1.5)
    ax.add_patch(box)
    ax.text(x, y, text, ha="center", va="center", fontsize=9)

# Arrows between levels.
for x1, x2 in [(2.9, 3.6), (6.4, 7.1)]:
    ax.annotate("", xy=(x2, 3.5), xytext=(x1, 3.5),
                arrowprops=dict(arrowstyle="->", lw=2, color="black"))

# Ground segment path.
path_items = [
    (1.5, 1.2, "SDO (GEO)\n~36,000 km", "#ffe0b2"),
    (4.0, 1.2, "Ka-band\n130 Mbps", "#fff9c4"),
    (6.5, 1.2, "White Sands\nGround Station", "#e0f7fa"),
    (9.0, 1.2, "Stanford JSOC\nArchive", "#f3e5f5"),
]
for x, y, text, color in path_items:
    box = FancyBboxPatch((x - 1.1, y - 0.6), 2.2, 1.2,
                         boxstyle="round,pad=0.1", facecolor=color,
                         edgecolor="black", lw=1)
    ax.add_patch(box)
    ax.text(x, y, text, ha="center", va="center", fontsize=8)

for x1, x2 in [(2.7, 2.9), (5.2, 5.4), (7.7, 7.9)]:
    ax.annotate("", xy=(x2, 1.2), xytext=(x1, 1.2),
                arrowprops=dict(arrowstyle="->", lw=1.5, color="gray"))

# (b) Data volume calculations.
ax = axes[1]

# Calculate raw data rate.
bits_per_image = 4096 * 4096 * 16  # 16-bit per pixel.
images_per_sec = 10 / 12  # 10 channels every 12 seconds.
raw_rate_mbps = bits_per_image * images_per_sec / 1e6

# Data volumes.
categories = ["Raw rate\n(Mbit/s)", "Compressed\n(Mbit/s)",
              "Daily\n(TB)", "Annual\n(TB)", "15-year\n(PB)"]
values = [raw_rate_mbps, 67, 1500, 500000, 7500000]
# Normalize to common scale for visualization.
display_vals = [raw_rate_mbps, 67, 1500, 500, 7.5]
units = ["Mbit/s", "Mbit/s", "TB", "TB", "PB"]
colors = ["#ef5350", "#66bb6a", "#42a5f5", "#ab47bc", "#ffa726"]

# Create two subpanels using twin axes for different scales.
labels_rate = ["Raw", "Rice-compressed"]
vals_rate = [raw_rate_mbps, 67]
labels_vol = ["Daily (TB)", "Annual (TB)", "15-year (PB)"]
vals_vol = [1.5, 500, 7500]

x1 = np.arange(len(labels_rate))
x2 = np.arange(len(labels_vol))

ax.bar(x1, vals_rate, color=["#ef5350", "#66bb6a"], edgecolor="black",
       width=0.5)
for i, v in enumerate(vals_rate):
    ax.text(i, v + 5, f"{v:.0f} Mbit/s", ha="center", fontsize=10,
            fontweight="bold")

# Second group shifted right.
offset = len(labels_rate) + 0.5
ax2 = ax.twinx()
ax2.bar(x2 + offset, vals_vol, color=["#42a5f5", "#ab47bc", "#ffa726"],
        edgecolor="black", width=0.5)
for i, v in enumerate(vals_vol):
    label_text = f"{v:.1f} TB" if v < 10 else f"{v:.0f} {'PB' if v > 5000 else 'TB'}"
    ax2.text(i + offset, v + 200, label_text, ha="center", fontsize=10,
             fontweight="bold")

ax.set_ylabel("Data Rate [Mbit/s]")
ax2.set_ylabel("Data Volume [TB or PB]")
ax.set_xticks(list(x1) + [x + offset for x in x2])
ax.set_xticklabels(labels_rate + labels_vol)
ax.set_title("AIA Data Volumes")

plt.tight_layout()
plt.show()

print(f"Raw data rate: {raw_rate_mbps:.0f} Mbit/s")
print(f"  = {4096}x{4096} x 16bit x 10ch / 12s")
print(f"Rice compression: ~3:1 -> ~67 Mbit/s")
print(f"Daily volume: ~1.5 TB (Level 0)")
print(f"Annual: ~500 TB")
print(f"15-year mission: ~7.5 PB cumulative")
print(f"\nComparison with SOHO EIT era:")
print(f"  EIT daily: ~10 MB -> AIA daily: ~1.5 TB")
print(f"  Ratio: {1.5e6 / 10:.0f}x increase in data volume")

---
## Part 6: EUV Imager Lineage Summary / EUV 영상기 계보 요약

논문 #9(EIT), #10(TRACE), #11(EUVI), #12(AIA)에서 다룬 EUV 태양 영상기의 전체 계보를 종합적으로 정리한다. EUI-HRI (Solar Orbiter)도 포함하여 미래 방향을 제시한다.

We provide a comprehensive summary of the EUV solar imager lineage covered in papers #9 (EIT), #10 (TRACE), #11 (EUVI), and #12 (AIA). We also include EUI-HRI (Solar Orbiter) to show the future direction.

In [ ]:
# Comprehensive comparison table data.
lineage = {
    "EIT":     {"year": 1995, "aperture": 12, "f_num": 14, "pixel": 2.6,
                "ccd": 1024, "fov": 45, "channels": 4, "cadence": 720,
                "data_day": 0.01, "orbit": "L1"},
    "TRACE":   {"year": 1998, "aperture": 30, "f_num": 29, "pixel": 0.5,
                "ccd": 1024, "fov": 8.5, "channels": 8, "cadence": 30,
                "data_day": 0.7, "orbit": "LEO"},
    "EUVI":    {"year": 2006, "aperture": 10, "f_num": 14, "pixel": 1.6,
                "ccd": 2048, "fov": 54, "channels": 4, "cadence": 150,
                "data_day": 0.05, "orbit": "L4/L5"},
    "AIA":     {"year": 2010, "aperture": 20, "f_num": 20, "pixel": 0.6,
                "ccd": 4096, "fov": 41, "channels": 10, "cadence": 12,
                "data_day": 1500, "orbit": "GEO"},
    "EUI-HRI": {"year": 2020, "aperture": 17, "f_num": 20, "pixel": 0.5,
                "ccd": 2048, "fov": 17, "channels": 2, "cadence": 2,
                "data_day": 1.0, "orbit": "helio"},
}

# Print the comparison table.
header = (f"{'':>10} {'EIT':>8} {'TRACE':>8} {'EUVI':>8} {'AIA':>8} "
          f"{'EUI-HRI':>8}")
print("=" * 60)
print("EUV Solar Imager Lineage / EUV 태양 영상기 계보")
print("=" * 60)
print(header)
print("-" * 60)

rows = [
    ("Year", "year", ""),
    ("Aperture", "aperture", "cm"),
    ("f/", "f_num", ""),
    ("Pixel", "pixel", '"'),
    ("CCD", "ccd", "^2"),
    ("FOV", "fov", "'"),
    ("Channels", "channels", ""),
    ("Cadence", "cadence", "s"),
    ("Data/day", "data_day", "GB"),
    ("Orbit", "orbit", ""),
]

for label, key, unit in rows:
    vals = []
    for name in ["EIT", "TRACE", "EUVI", "AIA", "EUI-HRI"]:
        v = lineage[name][key]
        if key == "orbit":
            vals.append(f"{v:>8}")
        elif key == "data_day":
            if v >= 1:
                vals.append(f"{v:>7.0f}{unit}")
            else:
                vals.append(f"{v:>7.2f}{unit}")
        elif key == "ccd":
            vals.append(f"{v:>5}{unit:>3}")
        elif isinstance(v, float):
            vals.append(f"{v:>7.1f}{unit}")
        else:
            vals.append(f"{v:>7}{unit}")
    print(f"{label:>10} {''.join(vals)}")

In [ ]:
# Radar / Spider chart comparing key metrics (normalized).
fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))

# Metrics to compare (higher = better for all after normalization).
metrics = ["Resolution", "FOV", "Channels", "Cadence", "CCD Size", "Aperture"]
n_metrics = len(metrics)

# Normalize: resolution (invert pixel scale), cadence (invert), others direct.
def normalize_metrics(params):
    """Normalize instrument metrics to [0, 1] for radar chart.

    Args:
        params: Dictionary of instrument parameters.

    Returns:
        List of normalized metric values.
    """
    # Max values for normalization.
    max_res = 1 / 0.5  # Best pixel = 0.5".
    max_fov = 54  # EUVI largest.
    max_ch = 10  # AIA.
    max_cad = 1 / 2  # EUI-HRI fastest.
    max_ccd = 4096  # AIA.
    max_ap = 30  # TRACE.

    return [
        (1 / params["pixel"]) / max_res,
        params["fov"] / max_fov,
        params["channels"] / max_ch,
        (1 / params["cadence"]) / max_cad,
        params["ccd"] / max_ccd,
        params["aperture"] / max_ap,
    ]


angles = np.linspace(0, 2 * np.pi, n_metrics, endpoint=False).tolist()
angles += angles[:1]  # Close the polygon.

colors_radar = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]
names_radar = ["EIT", "TRACE", "EUVI", "AIA", "EUI-HRI"]

for name, color in zip(names_radar, colors_radar):
    vals = normalize_metrics(lineage[name])
    vals += vals[:1]  # Close polygon.
    ax.plot(angles, vals, "o-", lw=2, color=color, label=name, markersize=5)
    ax.fill(angles, vals, alpha=0.1, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics, fontsize=10)
ax.set_ylim(0, 1.1)
ax.set_title("EUV Imager Capability Comparison (Normalized)",
             fontsize=13, pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=10)

plt.tight_layout()
plt.show()

print("Key observations / 주요 관찰:")
print("  - AIA has the most balanced and largest 'radar area' overall.")
print("  - TRACE excels in resolution & aperture but sacrifices FOV.")
print("  - EUI-HRI pushes cadence to extreme (2s) with high resolution.")
print("  - EIT was the pioneer — modest in all metrics but proved the concept.")
print("  - Each generation addressed the key limitation of its predecessor.")

---
## Summary / 요약

이 노트북에서 AIA의 핵심 설계와 성능을 분석했다:

This notebook analyzed AIA's key design and performance:

1. **온도 응답 함수 / Temperature Response**: 7개 EUV 채널의 온도 응답 함수를 재현하여 AIA의 온도 진단 능력을 시각화했다. 171 A가 가장 강한 응답을 보이며, 131 A와 193 A는 이중 피크를 가진다.

   Reproduced the 7 EUV channel response functions showing AIA's temperature diagnostic capability. 171 A has the strongest response; 131 A and 193 A show dual peaks.

2. **DEM 순방향 모델 / DEM Forward Model**: 합성 DEM을 사용하여 다른 태양 환경(조용한 태양, 활동영역, 플레어)에서의 채널별 "지문"이 어떻게 다른지 보여주었다.

   Used synthetic DEMs to demonstrate how channel fingerprints differ for quiet Sun, active regions, and flares.

3. **전임자 비교 / Predecessor Comparison**: AIA의 정보 처리량이 EIT 대비 약 22,000배 증가했으며, 높은 해상도와 전체 디스크 시야각을 동시에 달성했다.

   AIA's information throughput is ~22,000x EIT baseline, achieving both high resolution and full-disk FOV simultaneously.

4. **4-망원경 구조 / 4-Telescope Architecture**: 4개 망원경이 각각 2채널을 담당하여 12초 주기를 달성하는 설계를 시각화했다.

   Visualized how 4 telescopes each handling 2 channels achieve the 12-second cadence.

5. **데이터 파이프라인 / Data Pipeline**: 223 Mbit/s 원시 데이터가 Rice 압축 후 67 Mbit/s로 전송되며, 일일 1.5 TB, 15년 임무 기간 약 7.5 PB의 데이터를 생산한다.

   Raw 223 Mbit/s compressed to 67 Mbit/s, producing 1.5 TB/day and ~7.5 PB over the 15-year mission.

6. **EUV 영상기 계보 / EUV Imager Lineage**: EIT → TRACE → EUVI → AIA → EUI-HRI의 발전을 종합적으로 비교하여, 각 세대가 전임자의 핵심 한계를 극복했음을 보여주었다.

   Comprehensive comparison of EIT → TRACE → EUVI → AIA → EUI-HRI showing how each generation overcame its predecessor's key limitation.

---

**Paper**: Lemen, J.R., et al. (2012). "The Atmospheric Imaging Assembly (AIA) on the Solar Dynamics Observatory (SDO)." *Solar Physics*, 275, 17–40. [DOI: 10.1007/s11207-011-9776-8]